## 0. Imports

In [ ]:
import warnings

import torch

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

## 1. Path configuration

In [ ]:
from src.utils import DATA_DIR, DATA_EXTRACTED_DIR
print(DATA_DIR)
print(DATA_EXTRACTED_DIR)


## 2. Data loading

In [ ]:
from src.get_dataset_info import get_all_info
df_transcriptions, df_results, df_embeddings, df_features = get_all_info(transcribe=False, evaluate=False, embeddings=False, features= False)

In [ ]:
import os
import pandas as pd
df_results_without_label = df_results.drop(columns=["label"])

df = pd.merge(df_results_without_label, df_features, on="id", how="inner")
df = pd.merge(df, df_embeddings, on="id", how="inner")
df

### features + log-Mels

In [ ]:
from src.utils import DATA_DIR
print(DATA_DIR)


In [ ]:

from src.utils import get_patient_id,DATA_DIR, PATH_DIR_AUDIO_CONTROLS, PATH_DIR_AUDIO_PATIENTS, PATH_DIR_LABELS_CONTROLS, PATH_DIR_LABELS_PATIENTS

ids = []
labels = []
dict_id_labels  = {}
for grupo, dir_audio, dir_labels in [
    ("CONTROLS", PATH_DIR_AUDIO_CONTROLS, PATH_DIR_LABELS_CONTROLS),
    ("PATIENTS", PATH_DIR_AUDIO_PATIENTS, PATH_DIR_LABELS_PATIENTS),
]:  
    if grupo == "CONTROLS":
        label = 0
    else:
        label = 1
    for archivo_audio in sorted(os.listdir(dir_audio)):
        if not archivo_audio.endswith(".wav"):
            continue

        id_patient = get_patient_id(archivo_audio)
        ids.append(id_patient)
        labels.append(label)    
        dict_id_labels[id_patient] = label


## 3. Dataset combinado y CV-5 a nivel de individuo

In [ ]:
import numpy as np
from src.utils import create_5cv
folds = create_5cv(dict_id_labels)

from collections import Counter

print("Fold check (participant level):")

for i, f in enumerate(folds):

    index_train = f['train_ids']
    index_test  = f['test_ids']
    index_val   = f['val_ids']

    y_train = [int(np.squeeze(dict_id_labels[a])) for a in index_train]
    y_val   = [int(np.squeeze(dict_id_labels[a])) for a in index_val]
    y_test  = [int(np.squeeze(dict_id_labels[a])) for a in index_test]

    print(
        f"Fold {i+1}: "
        f"train={Counter(y_train)}  "
        f"val={Counter(y_val)}  "
        f"test={Counter(y_test)}"
    )

In [ ]:
import os
import warnings
from src.utils import get_patient_id
from src.audio_preprocessing import load_audio, parse_audio_segments, concat_audio, fragment_and_logmel

warnings.filterwarnings("ignore", category=UserWarning, module="torchaudio")

logmels_data = {}
for grupo, audio_dir, label_dir in [
    ("CONTROLS", PATH_DIR_AUDIO_CONTROLS, PATH_DIR_LABELS_CONTROLS),
    ("PATIENTS", PATH_DIR_AUDIO_PATIENTS, PATH_DIR_LABELS_PATIENTS),
]:  
    for audio_file in sorted(os.listdir(audio_dir)):
        if not audio_file.endswith(".wav"):
            continue

        id_patient = get_patient_id(audio_file)
        audio_path = os.path.join(audio_dir, audio_file)
        label_path   = os.path.join(label_dir, f"{id_patient}.txt")

        if not os.path.exists(label_path):
            print(f"  ⚠️  Sin etiquetas para {id_patient}, saltando.")
            continue

        print(f"  Processing: {audio_file}")
        segments = parse_audio_segments(label_path)

        audio = load_audio(audio_path, target_sr=16_000)
        patient_audio = concat_audio(audio, [(s,e) for s,e,t in segments])
        patient_logmel = fragment_and_logmel(patient_audio)
        logmels_data[id_patient] = patient_logmel

        

## 4. Arquitecturas

## 5. Funciones de entrenamiento y evaluación

## 6. Experimentos

### 6.1 Arquitectura 0 — AlzheimerCNN (baseline, solo log-Mel)

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, f1_score, recall_score, precision_score

from src.models import CNN, train_CNN, val_CNN, evaluate_by_individual, compute_ece, ModelWithTemperature, collate_fn

results = []
for fold_idx, fold in enumerate(folds):
    print(f"Fold {fold_idx+1}")

    ids_train = fold['train_ids']
    ids_val   = fold['val_ids']
    ids_test  = fold['test_ids']

    X_train, y_train, y_id_train = [], [], []
    X_val, y_val, y_id_val       = [], [], []
    X_test, y_test, y_id_test    = [], [], []

    for pid in ids_train:
        for frag in logmels_data[pid]:
            X_train.append(torch.tensor(frag, dtype=torch.float32))
            y_train.append(dict_id_labels[pid])
            y_id_train.append(pid)

    for pid in ids_val:
        for frag in logmels_data[pid]:
            X_val.append(torch.tensor(frag, dtype=torch.float32))
            y_val.append(dict_id_labels[pid])
            y_id_val.append(pid)

    for pid in ids_test:
        for frag in logmels_data[pid]:
            X_test.append(torch.tensor(frag, dtype=torch.float32))
            y_test.append(dict_id_labels[pid])
            y_id_test.append(pid)

    train_dataset = list(zip(X_train, y_train, y_id_train))
    val_dataset   = list(zip(X_val, y_val, y_id_val))
    test_dataset  = list(zip(X_test, y_test, y_id_test))

    train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True,collate_fn=collate_fn,num_workers=0,pin_memory=True)
    val_loader = DataLoader(val_dataset,batch_size=64,shuffle=False,collate_fn=collate_fn,num_workers=0,pin_memory=True)
    test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False,collate_fn=collate_fn,num_workers=0,pin_memory=True)

    model = CNN().to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    patience = 7
    best_loss = float('inf')
    patience_counter = 0

    for epoch in range(50):

        train_loss = train_CNN(model,train_loader,optimizer,criterion,device,epoch)

        val_loss = val_CNN(model,val_loader,criterion,device)

        print(f"Epoch {epoch+1} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

        scheduler.step(val_loss)

        if val_loss < best_loss:

            best_loss = val_loss
            patience_counter = 0

            best_weights = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }

        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered")
            break
    

    y_true, y_pred, y_prob = evaluate_by_individual(model, test_loader, device)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    prob_c1 = y_prob

    THRESH_CONF = 0.9

    clear_mask = (prob_c1 >= THRESH_CONF) | (prob_c1 <= (1 - THRESH_CONF))
    doubt_mask = ~clear_mask
    wrong_mask = y_true != y_pred

    acc_clear = np.mean(y_true[clear_mask] == y_pred[clear_mask]) if np.sum(clear_mask) > 0 else 0
    acc_doubt = np.mean(y_true[doubt_mask] == y_pred[doubt_mask]) if np.sum(doubt_mask) > 0 else 0

    print("\n" + "="*70)
    print("ANÁLISIS DE CONFIANZA (POR INDIVIDUO)")
    print("="*70)

    print(f"Claros (>{THRESH_CONF}): {np.sum(clear_mask)} | Acc: {acc_clear:.4f}")
    print(f"Dudosos (<{THRESH_CONF}): {np.sum(doubt_mask)} | Acc: {acc_doubt:.4f}")
    

    df_pred = pd.DataFrame({
        "id": np.array(ids_test),
        "y_true": y_true,
        "y_pred": y_pred,
        "prob_c1": prob_c1
    })

    print(df_pred)

    df_clear_wrong = pd.DataFrame({
        "id": np.array(ids_test)[clear_mask & wrong_mask],
        "y_true": y_true[clear_mask & wrong_mask],
        "y_pred": y_pred[clear_mask & wrong_mask],
        "prob_c1": prob_c1[clear_mask & wrong_mask]
    })
    
    if len(df_clear_wrong) > 0:
        print("\nCLAROS MAL CLASIFICADOS:")
        print(df_clear_wrong)

    df_doubt_wrong = pd.DataFrame({
        "id": np.array(ids_test)[doubt_mask & wrong_mask],
        "y_true": y_true[doubt_mask & wrong_mask],
        "y_pred": y_pred[doubt_mask & wrong_mask],
        "prob_c1": prob_c1[doubt_mask & wrong_mask]
    })
    if len(df_doubt_wrong) > 0:
        print("\nDUDOSOS MAL CLASIFICADOS:")
        print(df_doubt_wrong)

    ece = compute_ece(y_true, y_prob, n_bins=10)
    model_with_temp = ModelWithTemperature(model)
    model_with_temp.set_temperature(val_loader, device)
    y_true_temp, y_pred_temp, y_prob_temp = evaluate_by_individual(model_with_temp, test_loader, device)
    ece_temp = compute_ece(y_true_temp, y_prob_temp, n_bins=10)

    print(f"ECE: {ece:.4f}")
    print(f"ECE after temperature scaling: {ece_temp:.4f}")

    print("Classification Report:")
    print(classification_report(y_true, y_pred, digits=4))

    accuracy = np.mean(np.array(y_true) == np.array(y_pred))
    f1 = f1_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    precision = precision_score(y_true, y_pred, average='weighted')

    print(f"Fold {fold_idx+1} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, Recall: {recall:.4f}, Precision: {precision:.4f}")

    results.append({
        'fold': fold_idx + 1,
        'accuracy': accuracy,
        'f1_score': f1,
        'recall': recall,
        'precision': precision,
        'ece': ece,
        'ece_temp': ece_temp
    })

    model.load_state_dict(best_weights)

    os.makedirs("modelos", exist_ok=True)

    torch.save(model.state_dict(),f"modelos/model_fold_{fold_idx+1}.pth")

    
df_results = pd.DataFrame(results)
print(df_results)


In [ ]:
import pandas as pd
from sklearn.svm import SVC

def build_ensemble(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    ensemble_type: str = "stacking",   # "voting" | "stacking"
    cv_inner: int = 5,
    random_state: int = 42,
) -> object:
    """
    Entrena los modelos base con GridSearchCV, construye el ensemble
    del tipo indicado, lo ajusta sobre (X_train, y_train) y lo devuelve
    listo para llamar a predict_proba().

    Parameters
    ----------
    X_train      : features de entrenamiento
    y_train      : etiquetas de entrenamiento
    ensemble_type: "voting" | "stacking"
    cv_inner     : folds internos del GridSearchCV
    random_state : semilla

    Returns
    -------
    ensemble     : estimador sklearn ajustado con .predict_proba()
    """

    configs = {
        "svm": (
            SVC(kernel="linear", class_weight="balanced", probability=True),
            {"C": [0.01, 0.1, 1, 10, 100]},
        ),
        "svm_rbf": (
            SVC(kernel="rbf", class_weight="balanced", probability=True),
            {"C": [0.01, 0.1, 1, 10, 100]},
        ),
        "rf": (
            RandomForestClassifier(class_weight="balanced", random_state=random_state),
            {"n_estimators": [100, 200], "max_depth": [5, 10, None]},
        ),
        "lr": (
            LogisticRegression(class_weight="balanced", max_iter=5000),
            {"C": [0.01, 0.1, 1, 10, 100]},
        ),
        "lda": (
            LinearDiscriminantAnalysis(),
            {},
        ),
    }

    tuned_estimators = []

    for name, (model, param_grid) in configs.items():
        if param_grid:
            search = GridSearchCV(
                estimator=model,
                param_grid=param_grid,
                cv=cv_inner,
                scoring="f1",
                n_jobs=-1,
            )
            search.fit(X_train, y_train)
            best = search.best_estimator_
            print(f"  [{name}] best params: {search.best_params_}  "
                  f"(cv-f1={search.best_score_:.4f})")
        else:
            best = model   # LDA no tiene grid
            print(f"  [{name}] sin grid search")

        tuned_estimators.append((name, best))

    if ensemble_type == "voting":
        ensemble = VotingClassifier(
            estimators=tuned_estimators,
            voting="soft",
        )

    elif ensemble_type == "stacking":
        ensemble = StackingClassifier(
            estimators=tuned_estimators,
            final_estimator=LogisticRegression(
                class_weight="balanced", max_iter=5000
            ),
            cv=cv_inner,
            passthrough=False,
            n_jobs=-1,
        )

    else:
        raise ValueError(f"ensemble_type debe ser 'voting' o 'stacking', recibido: '{ensemble_type}'")

    ensemble.fit(X_train, y_train)
    print(f"\n✓ Ensemble '{ensemble_type}' ajustado.")

    return ensemble

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader

results = []

for fold_idx, fold in enumerate(folds):
    print(f"\n{'='*60}\nFOLD {fold_idx+1}\n{'='*60}")

    ids_train = fold['train_ids']
    ids_val   = fold['val_ids']
    ids_test  = fold['test_ids']

    X_train, y_train_mel, y_id_train = [], [], []
    X_val,   y_val_mel,   y_id_val   = [], [], []
    X_test,  y_test_mel,  y_id_test  = [], [], []

    for pid in ids_train:
        for frag in logmels_data[pid]:
            X_train.append(torch.tensor(frag, dtype=torch.float32))
            y_train_mel.append(dict_id_labels[pid])
            y_id_train.append(pid)

    for pid in ids_val:
        for frag in logmels_data[pid]:
            X_val.append(torch.tensor(frag, dtype=torch.float32))
            y_val_mel.append(dict_id_labels[pid])
            y_id_val.append(pid)

    for pid in ids_test:
        for frag in logmels_data[pid]:
            X_test.append(torch.tensor(frag, dtype=torch.float32))
            y_test_mel.append(dict_id_labels[pid])
            y_id_test.append(pid)

    train_dataset = list(zip(X_train, y_train_mel, y_id_train))
    val_dataset   = list(zip(X_val,   y_val_mel,   y_id_val))
    test_dataset  = list(zip(X_test,  y_test_mel,  y_id_test))

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                              collate_fn=collate_fn, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False,
                              collate_fn=collate_fn, num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False,
                              collate_fn=collate_fn, num_workers=0, pin_memory=True)

    train_features_raw = df.loc[list(ids_train)].drop(columns=['label']).values
    val_features_raw   = df.loc[list(ids_val)  ].drop(columns=['label']).values
    test_features_raw  = df.loc[list(ids_test) ].drop(columns=['label']).values

    y_train_feat = df.loc[list(ids_train), 'label'].values
    y_val_feat   = df.loc[list(ids_val),   'label'].values
    y_test_feat  = df.loc[list(ids_test),  'label'].values

    feat_scaler = StandardScaler().fit(train_features_raw)
    train_features = feat_scaler.transform(train_features_raw)
    val_features   = feat_scaler.transform(val_features_raw)
    test_features  = feat_scaler.transform(test_features_raw)

    cnn_model = CNN().to(device)
    cnn_model.load_state_dict(
        torch.load(f"modelos/model_fold_{fold_idx+1}.pth", map_location=device)
    )
    cnn_model.eval()

    y_true, y_pred_cnn, y_prob = evaluate_by_individual(cnn_model, test_loader, device)

    y_true       = np.array(y_true)
    y_pred_cnn   = np.array(y_pred_cnn)
    y_prob       = np.array(y_prob)
    ids_test_arr = np.array(list(ids_test))

    THRESH_CONF = 0.99
    clear_mask = (y_prob >= THRESH_CONF) | (y_prob <= (1 - THRESH_CONF))
    doubt_mask = ~clear_mask
    wrong_mask = y_true != y_pred_cnn

    print(f"\nClaros  (conf > {THRESH_CONF}): {clear_mask.sum()} individuos")
    print(f"Dudosos (conf < {THRESH_CONF}): {doubt_mask.sum()} individuos")

    best_f1_enet, best_enet_model = -1, None
    best_C, best_l1 = 1, 0.9

    for C in [0.1, 1, 10]:
        for l1_ratio in [0.5, 0.7, 0.9]:
            enet = LogisticRegression(
                penalty="elasticnet", solver="saga",
                C=C, l1_ratio=l1_ratio,
                max_iter=5000, n_jobs=1, random_state=42
            )
            enet.fit(train_features, y_train_feat)
            f1_val = f1_score(y_val_feat, enet.predict(val_features),
                              average="weighted", zero_division=0)
            if f1_val > best_f1_enet:
                best_f1_enet, best_enet_model = f1_val, enet
                best_C, best_l1 = C, l1_ratio

    print(f"\nElasticNet → C={best_C}, l1={best_l1}, F1_val={best_f1_enet:.4f}")

    coef_mask    = np.abs(best_enet_model.coef_.ravel()) > 1e-8
    selected_idx = np.where(coef_mask)[0]
    print(f"Selected variables: {len(selected_idx)} / {train_features.shape[1]}")

    X_train_sel = train_features[:, selected_idx]
    X_val_sel   = val_features[:,   selected_idx]
    X_test_sel  = test_features[:,  selected_idx]

    ensemble = build_ensemble(X_train_sel, y_train_feat, ensemble_type="stacking", cv_inner=5, random_state=42)

    y_pred_ensemble_all = ensemble.predict(X_test_sel)
    y_prob_ensemble_all = ensemble.predict_proba(X_test_sel)[:, 1]

    if doubt_mask.sum() > 0:
        doubt_indices = np.where(doubt_mask)[0]

        df_dudosos = pd.DataFrame({
            "id"           : ids_test_arr[doubt_indices],
            "clase_real"   : y_true[doubt_indices],
            "pred_CNN"     : y_pred_cnn[doubt_indices],
            "prob_CNN"     : np.round(y_prob[doubt_indices], 4),
            "pred_ensemble"     : y_pred_ensemble_all[doubt_indices],
            "prob_ensemble"     : np.round(y_prob_ensemble_all[doubt_indices], 4),
        })

        df_dudosos["CNN_ensemble_acuerdo"] = (
            df_dudosos["pred_CNN"] == df_dudosos["pred_ensemble"]
        )

        def resolve_doubt(row):
            if row["CNN_ensemble_acuerdo"]:
                return int(row["pred_CNN"])
            else:
                prob_media = (row["prob_CNN"] + row["prob_ensemble"]) / 2
                return int(prob_media >= 0.5)  # <0.5 → 0, >=0.5 → 1

        df_dudosos["new_pred"] = df_dudosos.apply(resolve_doubt, axis=1)

        print(f"\n── Individuos dudosos (fold {fold_idx+1}) ──")
        print(df_dudosos.to_string(index=False))
        df_dudosos.to_csv(f"dudosos_fold_{fold_idx+1}.csv", index=False)

        y_pred_hybrid = y_pred_cnn.copy()

        id_to_new_pred = dict(zip(df_dudosos["id"], df_dudosos["new_pred"]))

        for i, pid in enumerate(ids_test_arr):
            if pid in id_to_new_pred:
                y_pred_hybrid[i] = id_to_new_pred[pid]

    else:
        print("\nNo hay individuos dudosos en este fold.")
        df_dudosos    = pd.DataFrame()
        y_pred_hybrid = y_pred_cnn.copy()

    ece = compute_ece(y_true, y_prob, n_bins=10)
    model_with_temp = ModelelWithTemperature(cnn_model)
    model_with_temp.set_temperature(val_loader, device)
    y_true_temp, y_pred_temp, y_prob_temp = evaluate_by_individual(
        model_with_temp, test_loader, device
    )
    ece_temp = compute_ece(y_true_temp, y_prob_temp, n_bins=10)

    print(f"\nECE: {ece:.4f}  |  ECE tras Temperature Scaling: {ece_temp:.4f}")

    accuracy  = np.mean(y_true == y_pred_cnn)
    f1        = f1_score(y_true, y_pred_cnn, average='weighted', zero_division=0)
    recall    = recall_score(y_true, y_pred_cnn, average='weighted', zero_division=0)
    precision = precision_score(y_true, y_pred_cnn, average='weighted', zero_division=0)

    print("\nClassification Report (CNN pura):")
    print(classification_report(y_true, y_pred_cnn, digits=4))

    acc_hybrid  = np.mean(y_true == y_pred_hybrid)
    f1_hybrid   = f1_score(y_true, y_pred_hybrid, average='weighted', zero_division=0)
    rec_hybrid  = recall_score(y_true, y_pred_hybrid, average='weighted', zero_division=0)
    pre_hybrid  = precision_score(y_true, y_pred_hybrid, average='weighted', zero_division=0)

    print("\nClassification Report (Híbrida CNN + SVM en dudosos):")
    print(classification_report(y_true, y_pred_hybrid, digits=4))

    print(f"\n{'Métrica':<12} {'CNN pura':>10} {'Híbrida':>10}")
    print(f"{'Accuracy':<12} {accuracy:>10.4f} {acc_hybrid:>10.4f}")
    print(f"{'F1':<12} {f1:>10.4f} {f1_hybrid:>10.4f}")
    print(f"{'Recall':<12} {recall:>10.4f} {rec_hybrid:>10.4f}")
    print(f"{'Precision':<12} {precision:>10.4f} {pre_hybrid:>10.4f}")

    print("\nClassification Report (SVM sobre features):")
    print(classification_report(y_test_feat, y_pred_ensemble_all, digits=4))

    results.append({
        'fold'             : fold_idx + 1,
        'accuracy'         : accuracy,
        'f1_score'         : f1,
        'recall'           : recall,
        'precision'        : precision,
        'accuracy_hybrid'  : acc_hybrid,
        'f1_hybrid'        : f1_hybrid,
        'recall_hybrid'    : rec_hybrid,
        'precision_hybrid' : pre_hybrid,
        'ece'              : ece,
        'ece_temp'         : ece_temp,
        'n_dudosos'        : int(doubt_mask.sum()),
        'df_dudosos'       : df_dudosos,
    })

df_results = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'df_dudosos'}
    for r in results
])

print("\n── Results by fold ──")
compare_cols = ['fold',
                'f1_score', 'f1_hybrid',
                'recall',   'recall_hybrid',
                'precision','precision_hybrid',
                'accuracy', 'accuracy_hybrid',
                'ece',      'ece_temp',
                'n_dudosos']
print(df_results[compare_cols].to_string(index=False))

print("\n── Media ± Std ──")
numeric = df_results.drop(columns='fold')
print(numeric.agg(['mean', 'std']).T.to_string())

dfs_dudosos = [r['df_dudosos'] for r in results if len(r['df_dudosos']) > 0]
folds_con_dudosos = [r['fold'] for r in results if len(r['df_dudosos']) > 0]

if dfs_dudosos:
    df_todos_dudosos = (
        pd.concat(dfs_dudosos, keys=folds_con_dudosos)
        .reset_index(level=0)
        .rename(columns={'level_0': 'fold'})
    )
    print("\n── Todos los dudosos ──")
    print(df_todos_dudosos.to_string(index=False))
    df_todos_dudosos.to_csv("dudosos_todos_folds.csv", index=False)
else:
    print("\nNo hubo individuos dudosos en ningún fold.")